# Automatic Speech Recognition (ASR)

Welcome to this introductory guide to Automatic Speech Recognition (ASR)! ASR is a fascinating field within Artificial Intelligence (AI) that focuses on converting spoken language into written text. Think about voice assistants like Siri or Alexa, dictation software, or automatic captioning on videos – these are all powered by ASR technology.

**What is ASR?**

At its core, ASR systems take an audio waveform (your voice) as input and produce a sequence of words (text) as output. This process involves several complex steps, including:

1.  **Signal Processing:** Cleaning the audio signal, removing noise, and extracting relevant features.
2.  **Acoustic Modeling:** Mapping the audio features to basic units of sound, like phonemes /a/, /o/ ...
3.  **Language Modeling:** Understanding the probability of sequences of words occurring in a given language. This helps the system choose the most likely words.
4.  **Decoding:** Combining the acoustic and language models to find the most probable sequence of words corresponding to the input audio.

Modern ASR heavily relies on deep learning techniques, which have significantly improved accuracy over the past decade.

## Key ASR Models: Wav2Vec 2.0 and Whisper

Two prominent models have significantly advanced the field of ASR: Wav2Vec 2.0 and Whisper. These models leverage large amounts of data and sophisticated deep learning architectures to achieve state-of-the-art performance.

**Wav2Vec 2.0 (from Meta AI):** This model uses a clever approach called self-supervised learning. Instead of needing vast amounts of transcribed audio (audio paired with text), Wav2Vec 2.0 learns powerful representations directly from raw audio data. It masks parts of the audio input and tries to predict them based on the surrounding context, similar to how language models like BERT work with text. Once pre-trained on unlabeled audio, Wav2Vec 2.0 can be fine-tuned with a relatively small amount of labeled data for specific ASR tasks and languages, making it very versatile.

**Whisper (from OpenAI):** Whisper takes a different approach. It's trained on a massive and diverse dataset comprising 680,000 hours of multilingual and multitask supervised data collected from the web. This extensive training allows Whisper to perform remarkably well across a wide range of languages, accents, and noisy conditions, often without needing specific fine-tuning (a capability known as zero-shot performance). It's designed as an end-to-end system, directly mapping audio to text.

## Practical Examples: Arabic and Moroccan Darija ASR

Let's see how we can use pre-trained models for ASR tasks, specifically focusing on Arabic and Moroccan Darija. We will use models available on the Hugging Face Hub, a platform hosting thousands of pre-trained models.

First, we need to install the necessary libraries. If you haven't already, run the following cell. Note: Installation might take a few minutes, and Whisper might require `ffmpeg` to be installed on your system (`sudo apt update && sudo apt install ffmpeg`).

P.S: You don't necessarily need a GPU for this notebook.

In [ ]:
!pip install transformers torch soundfile librosa speechbrain
!sudo apt update && sudo apt install ffmpeg

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:6 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Fetched 3,917 B in 3s (1,143 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
7 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading pac

In [ ]:
!pip install -U datasets

### Example 1: Arabic ASR with Wav2Vec 2.0

We will use a Wav2Vec 2.0 model fine-tuned for Arabic. The `transformers` library from Hugging Face makes it easy to load and use these models. We'll need an audio file in Arabic to test this. For demonstration purposes, we'll load a sample from the `datasets` library, but you can replace `\'common_voice\' ` and the specific sample index with your own audio file path after loading it appropriately (e.g., using `librosa` or `soundfile`). Remember that the audio needs to be sampled at 16kHz for most Wav2Vec models.

In [ ]:
import torch
import librosa
from datasets import load_dataset, Audio
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

# Load a pre-trained Arabic ASR model and processor
model_name_wav2vec = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"
processor_wav2vec = Wav2Vec2Processor.from_pretrained(model_name_wav2vec) # Processor for Audio
model_wav2vec = Wav2Vec2ForCTC.from_pretrained(model_name_wav2vec)

# Load a sample Arabic audio file (e.g., from Common Voice dataset)
arabic_audio_sample = None
original_sentence = "(Could not load sample)"

# Load a small part of the dataset for demonstration
common_voice_ar = load_dataset("ayoubkirouane/Arabic_common_voice_11_0", split="train[:1%]")
# Resample the audio to 16kHz as required by the model
common_voice_ar = common_voice_ar.cast_column("audio", Audio(sampling_rate=16000))
# Select the first audio sample
arabic_audio_sample = common_voice_ar[0]["audio"]["array"]
sampling_rate = common_voice_ar[0]["audio"]["sampling_rate"]
original_sentence = common_voice_ar[0]['sentence']
print(f"Loaded sample audio with rate: {sampling_rate} Hz")

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Loaded sample audio with rate: 16000 Hz


In [ ]:
# Preprocess the audio
input_values = processor_wav2vec(arabic_audio_sample, sampling_rate=sampling_rate, return_tensors="pt").input_values

In [ ]:
# Perform inference
with torch.no_grad():
    logits = model_wav2vec(input_values).logits

In [ ]:
print(logits)

tensor([[[ 15.7974, -18.9776, -18.6947,  ...,  -6.8068,  -6.2553,  -6.4282],
         [ 15.7586, -19.1368, -18.8573,  ...,  -6.8094,  -6.1801,  -6.4290],
         [ 15.8280, -19.3175, -19.0202,  ...,  -6.8569,  -6.1166,  -6.4215],
         ...,
         [ 16.2255, -19.4670, -19.2203,  ...,  -6.7365,  -4.7465,  -5.8244],
         [ 15.8793, -19.2215, -18.9713,  ...,  -6.4134,  -4.3094,  -6.0376],
         [  2.3515,  -7.5080,  -7.3994,  ...,  -1.6064,  -1.3047,  -0.7052]]])


In [ ]:
# Decode the predicted IDs
predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor_wav2vec.batch_decode(predicted_ids)[0]

In [ ]:
print(f"Original Sentence (from dataset): {original_sentence}")
print(f"Wav2Vec Transcription: {transcription}")

Original Sentence (from dataset): عمي هو أخو أبي.
Wav2Vec Transcription: عمي هو أخ أبي


### Example 2: Multilingual ASR with Whisper

Whisper models are known for their strong multilingual capabilities out-of-the-box. We can easily use them via the `transformers` pipeline. This pipeline handles the pre-processing, model inference, and post-processing for us. We can test it on the same Arabic audio sample loaded earlier, or you can provide a path to any audio file (Whisper handles various formats if `ffmpeg` is installed). Whisper automatically detects the language, but you can also specify it for potentially better results.

In [ ]:
from transformers import pipeline
import numpy as np

# Load the ASR pipeline with a Whisper model
whisper_pipeline = pipeline("automatic-speech-recognition", model="openai/whisper-base")
print("Whisper pipeline loaded successfully.")

# Use the Arabic audio sample loaded in Example 1 if available
if arabic_audio_sample is not None:
    print("Transcribing Arabic sample with Whisper...")

    # 1. Ensure the audio is a 1D float32 numpy array
    audio_array = np.array(arabic_audio_sample, dtype=np.float32)
    if audio_array.ndim > 1:
        audio_array = np.squeeze(audio_array) # Flatten to 1D if it's nested
        audio_array = librosa.to_mono(audio_array) # Convert to mono if it's stereo


    # 2. Package it into a dictionary with the required sampling rate (16kHz)
    # Note: If your original audio is NOT 16kHz, you will need to resample it first
    # (e.g., using librosa.resample) before passing it here.
    audio_input_whisper = {
        "array": audio_array,
        "sampling_rate": 16000
    }

    # Perform transcription
    # result = whisper_pipeline(audio_input_whisper)
    result = whisper_pipeline(audio_input_whisper, chunk_length_s=30)
    transcription_whisper = result["text"]

    print(f"Original Sentence (from dataset): {original_sentence}")
    print(f"Whisper Transcription: {transcription_whisper}")

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


Whisper pipeline loaded successfully.
Transcribing Arabic sample with Whisper...


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


Original Sentence (from dataset): عمي هو أخو أبي.
Whisper Transcription:  عمي حوة حوة بي


### Example 3: Moroccan Darija ASR with SpeechBrain

For Moroccan Darija, we can use a model specifically trained for it, available through the `speechbrain` library, which also integrates with Hugging Face. This example demonstrates how to load the `speechbrain/asr-wav2vec2-dvoice-darija` model and use it for transcription. Similar to the previous example, you'll need a Darija audio file (sampled at 16kHz). We'll use a placeholder here; you should replace `'/content/path/to/your/darija_audio.wav' ` with the actual path to your audio file.

Challenge the model by intentionally creating a sample that lead to poor transcription results. Think about factors that could make transcription difficult.

In [ ]:
from speechbrain.pretrained import EncoderASR
import torchaudio
import os

# Load the pre-trained Darija ASR model
# This will download the model from Hugging Face Hub if not already cached
asr_model_sb = EncoderASR.from_hparams(source="speechbrain/asr-wav2vec2-dvoice-darija", savedir="pretrained_models/asr-wav2vec2-dvoice-darija")
print("Darija ASR model loaded successfully.")

/tmp/ipykernel_10749/3193684589.py:1: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import EncoderASR
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/pretrained_models/asr-wav2vec2-dvoice-darija/hyperparams.yaml'
/usr/lib/python3.12/importlib/__init__.py:90: UserWarning: Module 'speechbrain.lobes.models.huggingface_transformers' was deprecated, redirecting to 'speechbrain.integrations.huggingface'. Please update your script.
  return _bootstrap._gcd_import(name[level:], package, level)


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-large-xlsr-53
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:speechbrain.utils.fetching:Fetch wav2vec2.ckpt: Using symlink found at '/content/pretrained_models/asr-wav2vec2-dvoice-darija/wav2vec2.ckpt'
INFO:speechbrain.utils.fetching:Fetch asr.ckpt: Using symlink found at '/content/pretrained_models/asr-wav2vec2-dvoice-darija/asr.ckpt'
INFO:speechbrain.utils.fetching:Fetch tokenizer.ckpt: Using symlink found at '/content/pretrained_mod

Darija ASR model loaded successfully.


In [ ]:
darija_audio_file = '/content/path/to/your/darija_audio.wav'

# Perform transcription
print(f"Transcribing {darija_audio_file}...")
transcription_sb = asr_model_sb.transcribe_file(darija_audio_file)
print(f"SpeechBrain Transcription: {transcription_sb}")

INFO:speechbrain.utils.fetching:Fetch darija_audio.wav: Fetching from HuggingFace Hub '/content/path/to/your' if not cached


Transcribing /content/path/to/your/darija_audio.wav...


HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/path/to/your'. Use `repo_type` argument if needed.

## Finding and Using Moroccan Darija Datasets

While pre-trained models offer convenience, you may want to fine-tune a model on specific data (we’ll cover that in a different notebook). For Moroccan Darija, having access to the right datasets is essential. One great place to start is the Hugging Face Hub.

For example, the `speechbrain/asr-wav2vec2-dvoice-darija` model we used was trained on the **DVoice Darija** dataset, which is available [on Zenodo](https://zenodo.org/records/6342622) and may also have versions on Hugging Face.

Currently, two main datasets are commonly used for Darija:

* **DVoice** (available on Zenodo)
* **DODA**, which integrates smoothly with the Hugging Face datasets library and will be used for fine-tuning a Wav2Vec2 model in a separate notebook.

If your goal is to use ASR purely for inference, several pre-trained models for Darija are readily available:

* [`speechbrain/asr-wav2vec2-dvoice-darija`](https://huggingface.co/speechbrain/asr-wav2vec2-dvoice-darija)
* [`boumehdi/wav2vec2-large-xlsr-moroccan-darija`](https://huggingface.co/boumehdi/wav2vec2-large-xlsr-moroccan-darija)
* [`KandirResearch/Whisper-Small-Darija`](https://huggingface.co/KandirResearch/Whisper-Small-Darija)

or SoTA Recent model
* [`atlasia/moulsot.v0.3`](https://huggingface.co/atlasia/moulsot.v0.3)

For additional guidance or questions related to your project, feel free to contact **Yassine El Kheir**.


## 🧠 Understanding Check

1. Outline all the major steps to run inference on an exsiting ASR model.  
2. Identify two potential challenges specific to Darija ASR.